# CardioIA Vision - PBL Fase 4

Notebook consolidado para entrega do PBL FIAP.

Tema: Visao Computacional aplicada a raio-X de torax usando o dataset NIH Chest X-rays.

Problema principal:

- Classe 0: `No Finding`
- Classe 1: `Cardiomegaly`

Este notebook apresenta o fluxo completo: download, EDA, pre-processamento, splits, CNN propria, CNN padrao, Transfer Learning, Vision Transformer, comparacao, prototipo de inferencia e governanca/fairness.

**Aviso:** este projeto possui finalidade exclusivamente academica e educacional. Ele nao deve ser usado como ferramenta diagnostica real.

## 1. Configuracao inicial

Esta etapa carrega bibliotecas, caminhos do projeto e parametros globais. O codigo reutilizavel fica em `src/`, para manter o notebook limpo e facil de avaliar.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

config.ensure_project_directories()
config.seed_everything()

if HAS_SEABORN:
    sns.set_theme(style="whitegrid")

print(config.as_dict())

## 2. Download e organizacao do dataset

O dataset usado e o NIH Chest X-rays, recomendado no enunciado. Para executar automaticamente, configure credenciais do Kaggle. Se preferir, baixe manualmente e coloque os arquivos em:

```text
data/raw/Data_entry_2017.csv
data/raw/images/
```

O notebook separado `00_download_dataset_kaggle.ipynb` contem uma rotina mais detalhada de download e validacao. Aqui mantemos uma checagem objetiva para a entrega consolidada.

In [ ]:
CSV_PATH = config.RAW_DATA_DIR / config.DATA_ENTRY_FILENAME
IMAGE_INDEX_PATH = config.RAW_DATA_DIR / config.IMAGE_PATHS_FILENAME

if not CSV_PATH.exists():
    print("CSV ainda nao encontrado.")
    print("Opcao 1: execute notebooks/00_download_dataset_kaggle.ipynb.")
    print("Opcao 2: baixe pelo Kaggle e coloque Data_entry_2017.csv em data/raw/.")
else:
    print("CSV encontrado:", CSV_PATH)

image_count = sum(1 for path in config.RAW_IMAGES_DIR.rglob("*") if path.suffix.lower() in {".png", ".jpg", ".jpeg"})
print("Imagens locais encontradas:", image_count)
print("Indice de imagens existe:", IMAGE_INDEX_PATH.exists())

## 3. Analise exploratoria do dataset

A EDA verifica distribuicao de labels, casos multi-label, `No Finding`, `Cardiomegaly`, sexo, idade e posicao da imagem.

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError("Execute o download/organizacao do dataset antes da EDA.")

df = pd.read_csv(CSV_PATH)
df["labels_list"] = df["Finding Labels"].astype(str).str.split("|")
df["num_labels"] = df["labels_list"].apply(len)
df["is_no_finding"] = df["Finding Labels"].eq(config.NEGATIVE_LABEL)
df["has_cardiomegaly"] = df["labels_list"].apply(lambda labels: config.TARGET_LABEL in labels)
df["is_cardiomegaly_only"] = df["Finding Labels"].eq(config.TARGET_LABEL)
df["is_multilabel"] = df["num_labels"] > 1

eda_summary = pd.DataFrame({
    "metrica": ["linhas", "pacientes_unicos", "no_finding", "cardiomegaly_qualquer_label", "cardiomegaly_isolada", "multilabel"],
    "valor": [len(df), df["Patient ID"].nunique(), df["is_no_finding"].sum(), df["has_cardiomegaly"].sum(), df["is_cardiomegaly_only"].sum(), df["is_multilabel"].sum()],
})
eda_summary.to_csv(config.TABLES_DIR / "notebook_unico_eda_resumo.csv", index=False)
eda_summary

In [ ]:
labels_exploded = df[["Image Index", "labels_list"]].explode("labels_list").rename(columns={"labels_list": "label"})
label_distribution = labels_exploded["label"].value_counts().rename_axis("label").reset_index(name="image_count")
label_distribution.to_csv(config.TABLES_DIR / "notebook_unico_distribuicao_labels.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 7))
plot_data = label_distribution.sort_values("image_count", ascending=True)
ax.barh(plot_data["label"], plot_data["image_count"], color="#4C78A8")
ax.set_title("Distribuicao de labels no NIH Chest X-rays")
ax.set_xlabel("Quantidade de imagens")
ax.set_ylabel("Label")
plt.tight_layout()
fig.savefig(config.FIGURES_DIR / "notebook_unico_distribuicao_labels.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Dataset binario, balanceamento e splits por paciente

Criamos duas versoes: limpa e realista. Depois aplicamos undersampling e split por paciente para evitar vazamento de dados.

In [ ]:
from src import split_utils

df_with_paths = split_utils.attach_image_paths(df)
dataset_clean, dataset_realistic = split_utils.create_binary_datasets(df_with_paths)
dataset_clean_balanced = split_utils.undersample_balance(dataset_clean)
dataset_realistic_balanced = split_utils.undersample_balance(dataset_realistic)

dataset_clean.to_csv(config.SPLITS_DIR / "dataset_binary_clean.csv", index=False)
dataset_realistic.to_csv(config.SPLITS_DIR / "dataset_binary_realistic.csv", index=False)
dataset_clean_balanced.to_csv(config.SPLITS_DIR / "dataset_binary_clean_balanced_undersampled.csv", index=False)
dataset_realistic_balanced.to_csv(config.SPLITS_DIR / "dataset_binary_realistic_balanced_undersampled.csv", index=False)

split_utils.save_class_weights(split_utils.compute_class_weights(dataset_clean), config.SPLITS_DIR / "class_weights_clean.json")
split_utils.save_class_weights(split_utils.compute_class_weights(dataset_realistic), config.SPLITS_DIR / "class_weights_realistic.json")

patient_splits = split_utils.split_patient_ids(dataset_realistic_balanced)
split_frames = split_utils.apply_patient_split(dataset_realistic_balanced, patient_splits)
split_utils.save_split_frames(split_frames, config.SPLITS_DIR, prefix=None)
split_summary = split_utils.validate_patient_split(split_frames)
split_summary.to_csv(config.TABLES_DIR / "notebook_unico_resumo_splits.csv", index=False)
split_summary

## 5. Pre-processamento e DataLoaders

As imagens sao convertidas para RGB, redimensionadas para 224x224, normalizadas e carregadas em DataLoaders PyTorch.

In [ ]:
from src.datasets import DataLoaderConfig, create_dataloaders, inspect_batch

loader_config = DataLoaderConfig(
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
loaders = create_dataloaders(dataloader_config=loader_config)
batch = next(iter(loaders["train"]))
inspect_batch(batch)

## 6. Configuracao de treino

Por padrao, `RUN_TRAINING` vem como `False` para evitar iniciar treinos longos sem querer. Para executar o experimento final, altere para `True` no PC com GPU.

In [ ]:
from src.training.train import TrainConfig, train_model
from src.training.evaluate import evaluate_model, count_model_parameters

RUN_TRAINING = False
QUICK_EPOCHS = 1
FINAL_EPOCHS = config.DEFAULT_EPOCHS
EPOCHS = FINAL_EPOCHS

def train_and_evaluate(model, model_name: str, lr: float = config.LEARNING_RATE, epochs: int = EPOCHS):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    train_config = TrainConfig(model_name=model_name, epochs=epochs, device=config.DEVICE, metric_to_maximize="f1", use_amp=torch.cuda.is_available())
    train_result = train_model(model, loaders["train"], loaders["val"], criterion, optimizer, train_config)
    metrics = evaluate_model(model, loaders["test"], model_name=model_name, device=config.DEVICE, output_dir=config.METRICS_DIR, checkpoint_path=train_result["best_checkpoint_path"])
    return train_result, metrics

print("RUN_TRAINING:", RUN_TRAINING)
print("DEVICE:", config.DEVICE)

## 7. CNN propria

Esta e a rede criada pelo grupo, cumprindo o requisito de CNN do zero.

In [ ]:
from src.models.custom_cnn import create_custom_cnn

custom_cnn = create_custom_cnn()
print(count_model_parameters(custom_cnn))
with torch.no_grad():
    print("Forward CNN propria:", tuple(custom_cnn(torch.randn(2, 3, config.IMAGE_SIZE, config.IMAGE_SIZE)).shape))

if RUN_TRAINING:
    custom_result, custom_metrics = train_and_evaluate(custom_cnn, "cnn_propria_cardioia")
    display(custom_metrics)

## 8. CNN padrao

Baseline simples para comparar com a arquitetura propria.

In [ ]:
from src.models.standard_cnn import create_standard_cnn

standard_cnn = create_standard_cnn()
print(count_model_parameters(standard_cnn))
with torch.no_grad():
    print("Forward CNN padrao:", tuple(standard_cnn(torch.randn(2, 3, config.IMAGE_SIZE, config.IMAGE_SIZE)).shape))

if RUN_TRAINING:
    standard_result, standard_metrics = train_and_evaluate(standard_cnn, "cnn_padrao_baseline")
    display(standard_metrics)

## 9. Transfer Learning

Treinamos ResNet50, EfficientNetB0, EfficientNetB3 e DenseNet121. O fluxo completo com fase congelada e fine-tuning esta no notebook de apoio `07_treinamento_transfer_learning.ipynb`. Nesta entrega consolidada, mantemos a chamada principal e a validacao de criacao dos modelos.

In [ ]:
from src.models.transfer_learning import create_transfer_model, transfer_model_summary

TRANSFER_MODELS = ["resnet50", "efficientnet_b0", "efficientnet_b3", "densenet121"]

for model_name in TRANSFER_MODELS:
    model = create_transfer_model(model_name, pretrained=False, freeze_backbone=True)
    print(model_name, transfer_model_summary(model))
    if RUN_TRAINING:
        result, metrics = train_and_evaluate(model, f"{model_name}_head", lr=1e-3, epochs=3)
        display(metrics)

## 10. Vision Transformer

Modelo baseado em atencao para comparar com CNNs tradicionais.

In [ ]:
from src.models.vision_transformer import create_vision_transformer, transformer_model_summary

vit_model = create_vision_transformer("vit_b_16", pretrained=False, freeze_backbone=True)
print(transformer_model_summary(vit_model))
with torch.no_grad():
    print("Forward ViT:", tuple(vit_model(torch.randn(1, 3, config.IMAGE_SIZE, config.IMAGE_SIZE)).shape))

if RUN_TRAINING:
    vit_result, vit_metrics = train_and_evaluate(vit_model, "vit_b_16_head", lr=1e-3, epochs=3)
    display(vit_metrics)

## 11. Comparacao dos modelos e escolha final

Depois dos treinos, consolidamos metricas, comparamos desempenho/eficiencia e exportamos o melhor checkpoint.

In [ ]:
from src.training.comparison import load_metric_files, add_selection_scores, choose_final_model, export_final_checkpoint

all_metrics = load_metric_files(config.METRICS_DIR)
ranked_metrics = add_selection_scores(all_metrics)

if ranked_metrics.empty:
    print("Ainda nao ha metricas reais. Execute os treinos com RUN_TRAINING=True ou use os notebooks de treino dedicados.")
else:
    ranked_metrics.to_csv(config.METRICS_DIR / "model_comparison_ranked.csv", index=False)
    selected_model = choose_final_model(all_metrics)
    exported = export_final_checkpoint(selected_model, config.EXPORTED_MODELS_DIR)
    display(ranked_metrics[["model_name", "accuracy", "precision", "recall", "f1", "auc_roc", "seconds_per_image", "selection_score"]])
    print("Modelo final:", selected_model["model_name"])
    print("Checkpoint exportado:", exported)

## 12. Prototipo de inferencia

Com o modelo final exportado, podemos classificar uma imagem individual. O prototipo Flask tambem esta disponivel em `src/app/flask_app.py`.

In [ ]:
from src.inference import load_model_from_checkpoint, predict_image

IMAGE_PATH = None  # exemplo: "data/raw/images/00000001_000.png"

if IMAGE_PATH is None:
    print("Defina IMAGE_PATH para executar a inferencia no notebook.")
else:
    model, metadata = load_model_from_checkpoint(device=config.DEVICE)
    prediction = predict_image(IMAGE_PATH, model, metadata["model_name"], device=config.DEVICE)
    display(prediction)
    print("Este prototipo e exclusivamente academico e nao deve ser usado como diagnostico real.")

## 13. Governanca, fairness e limitacoes

A analise de fairness completa esta no notebook `11_governanca_fairness.ipynb`. Abaixo registramos os pontos obrigatorios para discussao.

In [ ]:
governance_points = [
    "O modelo nao substitui avaliacao medica.",
    "Os labels do dataset podem conter ruido, pois foram derivados de laudos.",
    "Falsos negativos podem atrasar investigacao clinica.",
    "Falsos positivos podem gerar preocupacao ou exames desnecessarios.",
    "A representatividade deve ser avaliada por sexo, idade e posicao da imagem.",
    "Antes de uso real, seriam necessarias validacao clinica e governanca formal.",
]

for point in governance_points:
    print("-", point)

## 14. Conclusao

Este notebook consolidado apresenta o pipeline completo solicitado no PBL: pre-processamento, CNN propria, Transfer Learning, prototipo, metricas, comparacao e discussao de governanca. Os notebooks separados permanecem no repositorio como apoio tecnico e execucao detalhada de cada etapa.